# Python — pytest: Writing Testable Pipeline Code
---

<div style="padding:15px;border-left:8px solid #1a8a5a;background:#e8f5e9;border-radius:4px;">
<strong>Core Insight:</strong> Untested pipelines fail silently in production.
The key pattern for data engineering: test pure transformation functions in isolation.
Don't test against live databases or S3 — mock external dependencies.
<code>pytest</code> is the standard. Fixtures avoid repetition. Parametrize catches edge cases.
</div>

### The Citi Context
At Citi, APM data collection pipelines ran nightly across 6,000 servers.
A bug in the CPU classification logic silently miscategorized "WARNING" servers as "OK"
for two weeks before anyone noticed. After adding pytest coverage with parametrized tests
for all threshold boundaries, that class of bug became impossible to ship.


In [ ]:
# ── 1. Basic pytest structure — functions, assertions, classes ───────────────
# Install: pip install pytest pandas
# Run: pytest test_transformations.py -v

# The functions we're testing (in transformations.py):

def compute_cpu_delta(today_cpu: float, yesterday_cpu: float) -> float:
    """Day-over-day CPU delta, rounded to 2 decimal places."""
    return round(today_cpu - yesterday_cpu, 2)


def classify_server(avg_cpu: float) -> str:
    """Classify server health based on average CPU utilization."""
    if avg_cpu >= 90:  return "CRITICAL"
    if avg_cpu >= 75:  return "WARNING"
    if avg_cpu >= 50:  return "NORMAL"
    return "IDLE"


# ── Tests: ────────────────────────────────────────────────────────────────────
import pytest

class TestCpuDelta:
    def test_positive_delta(self):
        assert compute_cpu_delta(80.5, 72.1) == 8.4

    def test_negative_delta(self):
        assert compute_cpu_delta(65.0, 72.5) == -7.5

    def test_zero_delta(self):
        assert compute_cpu_delta(72.5, 72.5) == 0.0

    def test_rounding(self):
        assert compute_cpu_delta(80.001, 70.002) == 9.999   # rounds correctly


class TestClassifyServer:
    # Parametrize: run the same test with multiple inputs — no copy-paste
    @pytest.mark.parametrize("cpu, expected", [
        (95.0,  "CRITICAL"),   # above threshold
        (90.0,  "CRITICAL"),   # exactly at threshold
        (89.9,  "WARNING"),    # just below CRITICAL
        (75.0,  "WARNING"),    # exactly at WARNING
        (74.9,  "NORMAL"),     # just below WARNING
        (50.0,  "NORMAL"),     # exactly at NORMAL
        (49.9,  "IDLE"),       # just below NORMAL
        (0.0,   "IDLE"),       # minimum
    ])
    def test_classification(self, cpu, expected):
        assert classify_server(cpu) == expected

# Run tests (simulated output — in real use, run pytest from terminal):
print("All threshold boundary tests pass if classify_server() is correct.")
print(classify_server(90.0), classify_server(89.9), classify_server(50.0))


In [ ]:
# ── 2. Fixtures — shared test data and setup ─────────────────────────────────
import pytest
import pandas as pd

@pytest.fixture
def sample_metrics_df():
    """
    Reusable DataFrame fixture — injected by pytest into any test that names it.
    Created ONCE per test (scope="function" by default).
    For expensive fixtures (SparkSession): scope="session" creates once per test run.
    """
    return pd.DataFrame({
        "server_id":   ["srv-01", "srv-02", "srv-03", "srv-04"],
        "avg_cpu":     [85.0,      45.2,     92.1,     75.0],
        "region":      ["us-east", "us-east", "eu-west", "eu-west"],
        "report_date": ["2026-02-26"] * 4
    })


def test_high_cpu_filter(sample_metrics_df):
    high_cpu = sample_metrics_df[sample_metrics_df["avg_cpu"] >= 80]
    assert len(high_cpu) == 2
    assert set(high_cpu["server_id"]) == {"srv-01", "srv-03"}


def test_critical_servers(sample_metrics_df):
    critical = sample_metrics_df[sample_metrics_df["avg_cpu"] >= 90]
    assert len(critical) == 1
    assert critical["server_id"].iloc[0] == "srv-03"


def test_mean_cpu(sample_metrics_df):
    mean = sample_metrics_df["avg_cpu"].mean()
    assert abs(mean - 74.325) < 0.001   # floating point: compare with tolerance


def test_region_groupby(sample_metrics_df):
    grouped = sample_metrics_df.groupby("region")["avg_cpu"].mean()
    assert abs(grouped["us-east"] - 65.1) < 0.01
    assert abs(grouped["eu-west"] - 83.55) < 0.01

# Simulate fixture usage (real use: just run `pytest`)
df = sample_metrics_df()   # call directly for demonstration
print(df)
print("High CPU servers:", df[df["avg_cpu"] >= 80]["server_id"].tolist())


In [ ]:
# ── 3. Mocking external dependencies — S3, databases, APIs ──────────────────
from unittest.mock import patch, MagicMock
import io, pandas as pd

# The function that calls S3 (we do NOT want actual S3 in tests)
def load_metrics_from_s3(bucket: str, key: str) -> pd.DataFrame:
    import boto3
    s3 = boto3.client("s3")
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(obj["Body"])


# ── Test it without touching S3 ───────────────────────────────────────────────
@patch("boto3.client")
def test_load_metrics_from_s3(mock_boto3):
    # Build mock response: fake CSV content as bytes
    csv_content = b"server_id,avg_cpu,report_date
srv-01,72.5,2026-02-26
srv-02,45.2,2026-02-26"

    mock_s3 = MagicMock()
    mock_boto3.return_value = mock_s3
    mock_s3.get_object.return_value = {"Body": io.BytesIO(csv_content)}

    # Call the real function — it uses the mocked boto3
    result = load_metrics_from_s3("my-bucket", "metrics/daily.csv")

    # Assert on result
    assert len(result) == 2
    assert result["avg_cpu"].iloc[0] == 72.5
    assert list(result.columns) == ["server_id", "avg_cpu", "report_date"]

    # Assert the mock was called correctly
    mock_s3.get_object.assert_called_once_with(Bucket="my-bucket", Key="metrics/daily.csv")

# Run the mock test directly (pytest would do this automatically)
test_load_metrics_from_s3()
print("Mock test passed — S3 was never called.")


In [ ]:
# ── 4. Testing pipeline transformations — pure functions are the sweet spot ──
import pandas as pd

# ❌ Anti-pattern: testing too much at once (integration test disguised as unit test)
# def test_full_pipeline():
#     run_pipeline()   # hits DB, S3, Spark — slow, fragile, not a unit test

# ✅ Pattern: decompose pipeline into pure transformation functions, test each

def filter_production_servers(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only production environment servers."""
    return df[df["environment"] == "prod"].copy()


def add_alert_column(df: pd.DataFrame) -> pd.DataFrame:
    """Add alert_level column based on avg_cpu thresholds."""
    def classify(cpu):
        if cpu >= 90: return "CRITICAL"
        if cpu >= 75: return "WARNING"
        return "OK"
    df = df.copy()
    df["alert_level"] = df["avg_cpu"].apply(classify)
    return df


def aggregate_by_region(df: pd.DataFrame) -> pd.DataFrame:
    """Compute avg and max CPU per region."""
    return (df.groupby("region")
              .agg(avg_cpu=("avg_cpu", "mean"),
                   max_cpu=("avg_cpu", "max"),
                   server_count=("server_id", "count"))
              .reset_index())


# ── Unit tests for each pure function ─────────────────────────────────────────
def test_filter_production_servers():
    df = pd.DataFrame({
        "server_id": ["s1", "s2", "s3"],
        "environment": ["prod", "dev", "prod"],
        "avg_cpu": [80.0, 30.0, 90.0]
    })
    result = filter_production_servers(df)
    assert len(result) == 2
    assert set(result["server_id"]) == {"s1", "s3"}
    assert all(result["environment"] == "prod")


def test_add_alert_column_boundaries():
    df = pd.DataFrame({"avg_cpu": [95.0, 90.0, 89.9, 75.0, 74.9, 0.0], "server_id": list("abcdef")})
    result = add_alert_column(df)
    expected = ["CRITICAL", "CRITICAL", "WARNING", "WARNING", "OK", "OK"]
    assert list(result["alert_level"]) == expected


def test_aggregate_by_region():
    df = pd.DataFrame({
        "server_id": ["s1", "s2", "s3"],
        "region": ["east", "east", "west"],
        "avg_cpu": [70.0, 80.0, 90.0]
    })
    result = aggregate_by_region(df)
    east = result[result["region"] == "east"].iloc[0]
    assert abs(east["avg_cpu"] - 75.0) < 0.01
    assert east["max_cpu"] == 80.0
    assert east["server_count"] == 2


# Run all tests
test_filter_production_servers()
test_add_alert_column_boundaries()
test_aggregate_by_region()
print("All pipeline transformation tests passed.")


## 🎤 5 Interview Q&A

**Q1: What is the difference between unit tests and integration tests?**

Unit tests test ONE function in isolation, with all dependencies mocked or stubbed.
They run in milliseconds and catch logic bugs.
Integration tests test multiple components working together with real (or near-real) external systems.
They catch wiring bugs — two components that work individually but break when combined.
In data engineering: unit test your transformation functions (pure, no I/O).
Integration test your end-to-end pipeline with a test database or local filesystem.

---

**Q2: What is `@pytest.fixture` and why use it instead of class setUp?**

A fixture is a factory that creates shared test resources and handles teardown automatically.
Unlike `setUp`, fixtures are injected by name — a test only gets the fixtures it names in its parameters.
This avoids creating expensive resources (SparkSession, DB connection) for tests that don't need them.
`scope="session"` creates the resource once for the entire test run — critical for SparkSession startup cost.

---

**Q3: When should you mock vs when should you test against the real thing?**

Mock when: the dependency is external (S3, database, API), slow, or has side effects.
Test against real when: you're writing an integration test and the real behavior matters.
Rule: unit tests should NEVER make network calls or write to disk.
Mock at the boundary where your code calls external code — use `@patch("boto3.client")`,
not a fake S3 service.

---

**Q4: What is `@pytest.mark.parametrize` and when is it essential?**

Parametrize runs the same test with multiple input/expected pairs, defined as a list of tuples.
Essential when: testing threshold boundaries (cpu=89.9 → WARNING, cpu=90.0 → CRITICAL).
Without it: you either write 8 identical test functions, or test only one case and miss boundaries.
Boundary testing is where most bugs hide — parametrize makes it painless to be thorough.

---

**Q5: How do you test a Spark job without running a full cluster?**

Use a local SparkSession in tests: `SparkSession.builder.master("local[1]").getOrCreate()`.
Test transformation functions with small DataFrames (10-100 rows).
Keep Spark tests in a separate test suite — they take 5-10 seconds to start.
Use `scope="session"` fixture so SparkSession starts once per test run, not once per test.
```python
@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.master("local[1]").appName("test").getOrCreate()
```


## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Unit Test** | Tests one function in isolation — no external calls, all dependencies mocked |
| **Integration Test** | Tests multiple components together with real (or near-real) external systems |
| **Fixture** | pytest factory that creates shared test resources and handles teardown |
| **Parametrize** | Run the same test with multiple inputs — `@pytest.mark.parametrize` |
| **Mock** | A fake object that replaces a dependency in tests — `unittest.mock.MagicMock` |
| **Patch** | Temporarily replace a module attribute during a test — `@patch("module.attribute")` |
| **Assertion** | A statement that must be true — `assert result == expected` |
| **Test Scope** | How long a fixture lives: function (default), class, module, session |
| **Pure Function** | No side effects, same output for same input — the ideal unit for testing |
| **Test Coverage** | % of code lines executed during test run — aim for >80% on core logic |
| **Boundary Test** | Testing at exact threshold values — where off-by-one errors hide |
| **Test Suite** | A collection of tests — often split into unit and integration suites |


## 💼 The Citi Narrative

**Context:** The Citi APM pipeline classified servers into CRITICAL/WARNING/NORMAL/IDLE based
on average CPU. This classification drove automated Jira ticket creation for capacity review.

**The Bug:** For two weeks, servers at exactly 75.0% CPU were classified as "NORMAL" instead
of "WARNING" — a one-character bug (`>` instead of `>=`). No tests existed.
12 servers that needed capacity review were silently dropped from the Jira queue.

**The Fix — Adding pytest:**
1. Wrote parametrized tests for all 8 threshold boundary values
2. The bug was caught immediately on the first test run
3. Added fixtures for the standard test DataFrame (3 servers, one per tier)
4. Added mock tests for the S3 loader — tests now ran in 0.2 seconds instead of 30 seconds (no S3 calls)

**The Outcome:** After adding pytest coverage, zero classification bugs shipped in the next 6 months.
Test suite ran in CI in < 10 seconds. Every PR required green tests before merge.

**The Lesson:** The most common data pipeline bugs are boundary bugs in classification logic.
Parametrized tests at threshold values (exactly at, just below, just above) catch them
before they reach production.

**Interview Line:** *"At Citi we had a silent bug in CPU classification — servers at exactly 75%
weren't getting WARNING tickets. Two weeks, no one noticed. After adding parametrized pytest tests
for every threshold boundary, that class of bug became impossible to ship. Test run: 0.2 seconds.
Bug detection: immediate."*
